In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import glob
import pickle
import csv

In [ ]:
γ_list     = ["0.079477","0.25133","0.79477","2.5133","7.9477","25.133","79.477","251.33"]
Q          = [316, 100, 31.6, 10, 3.16, 1, 0.316, 0.1]
all_states = [f"B{i}" for i in range(1,5)] + [f"D{i}" for i in range(1,11)]

In [ ]:
def latest_file(pattern):
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No file matching: {pattern}")
    print(files[-1])
    return files[-1]

In [ ]:
def max_tracedist_table(file):
    """For each gamma compute max TrDist over all states & time points.
    Returns a list of dicts with keys:
      gamma, Q, max_TraceDist, state, time_index, time_value.
    """
    rows = []
    for i, γ_i in enumerate(γ_list):
        global_max = -np.inf
        best_state = None
        best_tidx  = None
        best_tval  = None
        with h5py.File(file, "r") as f:
            for state in all_states:
                td  = f[γ_i][state]["TraceDist"][...]
                t   = f[γ_i][state]["time"][...]
                idx = int(np.argmax(td))
                if td[idx] > global_max:
                    global_max = float(td[idx])
                    best_state = state
                    best_tidx  = idx
                    best_tval  = float(t[idx])
        rows.append({"gamma": γ_i, "Q": Q[i],
                     "max_TraceDist": global_max,
                     "state": best_state,
                     "time_index": best_tidx,
                     "time_value": best_tval})
    return rows

In [ ]:
kossak_rows   = max_tracedist_table(latest_file("E_KOSSAK_TRACEDIST_ALLSTATES_*.h5"))
lindblad_rows = max_tracedist_table(latest_file("E_LINDBLAD_TRACEDIST_ALLSTATES_*.h5"))

with open("NonMark.pkl", "rb") as fh:
    NonMark = pickle.load(fh)

In [ ]:
# Print argmax table
col = "{:>10}  {:>6}  {:>12}  {:>12}  {:>6}  {:>6}  {:>8}"
header = col.format("gamma", "Q", "ansatz", "max_TrDist", "state", "t_idx", "t_val")
print(header)
print("-" * len(header))
for lrow, krow in zip(lindblad_rows, kossak_rows):
    for ansatz, row in [("Lindblad", lrow), ("Kossak", krow)]:
        print(col.format(
            row["gamma"], row["Q"], ansatz,
            f"{row['max_TraceDist']:.6f}",
            row["state"], row["time_index"],
            f"{row['time_value']:.3f}"))

In [ ]:
# Combined curve: Lindblad for high-Q (indices 0-3), Kossak for low-Q (4-7)
combined_max = ([r["max_TraceDist"] for r in lindblad_rows[:4]] +
                [r["max_TraceDist"] for r in kossak_rows[4:]])

plt.rcParams.update({"font.size": 14, "axes.labelsize": 16,
                     "xtick.labelsize": 14, "ytick.labelsize": 14,
                     "legend.fontsize": 14})

fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(range(1, 9), combined_max, marker="o", color="#009E73",
         linewidth=1.5,
         label=r"max $T(\rho_{\mathrm{exact}},\rho_{\mathrm{SID}})$")

ax2 = ax1.twinx()
ax2.plot(range(1, 9), NonMark, marker="+", color="red", linewidth=1.5,
         label=r"$\mathcal{N}$, non-Markovianity")
ax2.set_yscale("log"); ax2.set_ylim(1e-6, .9)
ax2.set_ylabel(r"$\mathcal{N}$, non-Markovianity measure", color="red")
ax2.tick_params(axis="y", colors="red")
ax2.spines["right"].set_color("red")
ax2.legend(loc=1)

ax1.set_yscale("log"); ax1.set_ylim(1e-6, 1.0)
ax1.set_xticks(range(1, 9), Q)
ax1.set_xlabel(r"$Q=\nu/\gamma$, quality factor")
ax1.set_ylabel(
    r"$\max T(\rho_{\mathrm{exact}},\rho_{\mathrm{SID}})$, max trace distance")
ax1.legend(loc=2)

plt.tight_layout()
plt.show()
fig.savefig("SB_SID_MAX_TRACEDIST_vs_nonMarkovianity.PDF")

In [ ]:
# Save combined argmax table as CSV
csv_rows = []
for row in lindblad_rows[:4]:
    csv_rows.append({**row, "ansatz": "Lindblad"})
for row in kossak_rows[4:]:
    csv_rows.append({**row, "ansatz": "Kossak"})

fieldnames = ["gamma", "Q", "ansatz", "max_TraceDist", "state", "time_index", "time_value"]
with open("max_tracedist_argmax.csv", "w", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(csv_rows)
print("Saved: max_tracedist_argmax.csv")